# GGUF Export — Kaggle

Loads the LoRA adapter from HF Hub, merges weights into the base model,
saves the merged model to `/kaggle/working/` (persists across cell runs),
then converts to GGUF and pushes to HF Hub.

**Run cells in order. Cell 4 (merge) takes ~10 min. Cell 5 (GGUF) takes ~20 min.**

In [ ]:
!pip install --quiet --upgrade unsloth bitsandbytes

In [ ]:
import os

HF_USERNAME     = "CopyleftCultivars"
HF_ADAPTER_REPO = f"{HF_USERNAME}/gemma4-natural-farming-lora"
HF_GGUF_REPO    = f"{HF_USERNAME}/gemma4-natural-farming-gguf"
MAX_SEQ_LENGTH  = 1024

# List top-level datasets only — avoids symlink recursion inside Kaggle mounts
print("=== /kaggle/input datasets ===")
try:
    for name in sorted(os.listdir("/kaggle/input")):
        path = f"/kaggle/input/{name}"
        try:
            contents = os.listdir(path)
            print(f"  {name}/  ({len(contents)} items: {contents[:4]})")
        except Exception as e:
            print(f"  {name}/  (unreadable: {e})")
except Exception as e:
    print(f"  Could not list /kaggle/input: {e}")

# Update this to match the exact folder name printed above
KAGGLE_INPUT = "/kaggle/input/merged-adaptor-gemma4"

# Handle zip that extracted into a subfolder
if os.path.exists(KAGGLE_INPUT):
    try:
        subdirs = [d for d in os.listdir(KAGGLE_INPUT)
                   if os.path.isdir(os.path.join(KAGGLE_INPUT, d))]
        if subdirs:
            KAGGLE_INPUT = os.path.join(KAGGLE_INPUT, subdirs[0])
            print(f"\nSubfolder detected — using: {KAGGLE_INPUT}")
    except Exception:
        pass

if os.path.exists(KAGGLE_INPUT) and os.listdir(KAGGLE_INPUT):
    MERGED_DIR = KAGGLE_INPUT
    print(f"Using pre-merged model: {MERGED_DIR}")
else:
    MERGED_DIR = "/kaggle/working/gemma4-merged"
    print(f"\n'{KAGGLE_INPUT}' not found — merge cell will save to: {MERGED_DIR}")

GGUF_TEMP = "/kaggle/working"
print(f"GGUF repo: {HF_GGUF_REPO}")

In [ ]:
from huggingface_hub import login
import getpass
token = getpass.getpass("HuggingFace write token: ")
login(token=token)
print("Logged in.")

In [ ]:
# Patch subprocess so llama.cpp errors print live instead of being
# swallowed by Unsloth's capture_output=True.
# Only applied to calls that don't already have stdout/stderr set.
import subprocess
_orig_run = subprocess.run
def _loud_run(args, **kwargs):
    check = kwargs.pop("check", False)
    # Only intercept if nothing else is already capturing output
    if "stdout" not in kwargs and "stderr" not in kwargs:
        kwargs.pop("capture_output", None)
        result = _orig_run(args, capture_output=True, **kwargs)
        if result.returncode != 0:
            dec = lambda b: b.decode(errors="replace") if isinstance(b, bytes) else (b or "")
            print("\n=== llama.cpp STDOUT ===")
            print(dec(result.stdout)[-4000:] or "(empty)")
            print("=== llama.cpp STDERR ===")
            print(dec(result.stderr)[-4000:] or "(empty)")
    else:
        result = _orig_run(args, **kwargs)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, args,
                                            getattr(result, "stdout", None),
                                            getattr(result, "stderr", None))
    return result
subprocess.run = _loud_run
print("Subprocess patch active.")

In [ ]:
# Load base model + LoRA adapter, merge, save to MERGED_DIR.
# Saved to disk so the GGUF cell can run independently if this session resets.
import os, torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if os.path.exists(MERGED_DIR) and os.listdir(MERGED_DIR):
    print(f"Merged model already at {MERGED_DIR} — skipping merge step.")
else:
    print(f"Loading adapter from {HF_ADAPTER_REPO} ...")
    model, tokenizer = FastModel.from_pretrained(
        model_name=HF_ADAPTER_REPO,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,    # bfloat16 avoids bitsandbytes device_map issues
        dtype=torch.bfloat16,  # ~8GB VRAM for 4B model, fits on T4
    )
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-it")

    print(f"Saving merged model to {MERGED_DIR} ...")
    model.save_pretrained_merged(
        MERGED_DIR,
        tokenizer,
        save_method="merged_16bit",
    )
    print("Merge complete.")

import subprocess
result = subprocess.run(["du", "-sh", MERGED_DIR], capture_output=True, text=True)
print(f"Merged model size: {result.stdout.strip()}")

In [ ]:
import gc, subprocess, pathlib, os
import torch
from huggingface_hub import HfApi

LLAMA_DIR  = pathlib.Path("/root/.unsloth/llama.cpp")
CONVERTER  = LLAMA_DIR / "convert_hf_to_gguf.py"
QUANTIZE   = LLAMA_DIR / "build/bin/llama-quantize"
GGUF_F16   = pathlib.Path(GGUF_TEMP) / "gemma4.F16.gguf"
GGUF_Q4    = pathlib.Path(GGUF_TEMP) / "gemma4.Q4_K_M.gguf"

# Free the model from memory BEFORE conversion — the converter only reads
# from disk, so we don't need the model loaded. This frees ~8GB RAM/VRAM.
try:
    del model, tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Step 1: safetensors → F16 GGUF
if not GGUF_F16.exists():
    print("Converting to F16 GGUF (reads from disk, ~3 min) ...")
    subprocess.run([
        "python3", str(CONVERTER),
        "--outfile", str(GGUF_F16),
        "--outtype", "f16",
        MERGED_DIR,
    ], check=True)
    print(f"F16 GGUF: {GGUF_F16} ({GGUF_F16.stat().st_size/1e9:.1f} GB)")
else:
    print(f"F16 GGUF already exists: {GGUF_F16}")

# Step 2: F16 GGUF → Q4_K_M
if not GGUF_Q4.exists():
    print("Quantizing to Q4_K_M (~5 min) ...")
    subprocess.run([str(QUANTIZE), str(GGUF_F16), str(GGUF_Q4), "Q4_K_M"],
                   check=True)
    print(f"Q4_K_M GGUF: {GGUF_Q4} ({GGUF_Q4.stat().st_size/1e9:.1f} GB)")
else:
    print(f"Q4_K_M GGUF already exists: {GGUF_Q4}")

# Step 3: upload to HF Hub
print("Uploading to HF Hub ...")
api = HfApi()
api.create_repo(repo_id=HF_GGUF_REPO, exist_ok=True, private=False)
api.upload_file(
    path_or_fileobj=str(GGUF_Q4),
    path_in_repo=GGUF_Q4.name,
    repo_id=HF_GGUF_REPO,
)
print(f"Done — https://huggingface.co/{HF_GGUF_REPO}")

In [ ]:
# Build llama-quantize and place it where unsloth's check_llama_cpp() searches.
import subprocess, os, pathlib, shutil, inspect
import unsloth_zoo.llama_cpp as _uz

# Show what paths check_llama_cpp actually searches
src = inspect.getsource(_uz.check_llama_cpp)
print("=== check_llama_cpp source (first 60 lines) ===")
print("\n".join(src.splitlines()[:60]))

LLAMA_DIR = pathlib.Path("/root/.unsloth/llama.cpp")
QUANTIZE  = LLAMA_DIR / "build/bin/llama-quantize"

if not LLAMA_DIR.exists():
    print("\nCloning llama.cpp ...")
    LLAMA_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/ggerganov/llama.cpp",
                    str(LLAMA_DIR)], check=True)

if not QUANTIZE.exists():
    print("\nBuilding llama-quantize (~3 min) ...")
    build = LLAMA_DIR / "build"
    build.mkdir(exist_ok=True)
    subprocess.run(["cmake", "..", "-DCMAKE_BUILD_TYPE=Release",
                    "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"],
                   cwd=str(build), check=True)
    subprocess.run(["cmake", "--build", ".", "--target", "llama-quantize",
                    f"-j{os.cpu_count() or 4}"],
                   cwd=str(build), check=True)

# Copy binary to every plausible location unsloth might check
for dest in [LLAMA_DIR / "llama-quantize",
             LLAMA_DIR / "build" / "llama-quantize",
             LLAMA_DIR / "build" / "Release" / "llama-quantize"]:
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(QUANTIZE), str(dest))
        os.chmod(str(dest), 0o755)
        print(f"Copied to: {dest}")

# Show all executables under llama.cpp so we can verify
print("\nExecutables under /root/.unsloth/llama.cpp:")
for p in sorted(LLAMA_DIR.rglob("llama-quantize*")):
    print(f"  {p}")